# 🌬️ AirScope SensorLab — RAG01 建立向量資料庫

學校：中原大學｜系級：＿＿＿＿｜學號：＿＿＿＿｜姓名：＿＿＿＿

這份程式從官方 PDF 讀入文字，分塊、建立真正的 FAISS 向量索引，最後輸出 **`rag-data.zip`**。
不需要 Groq 或 Hugging Face 金鑰。[前往 RAG02](https://colab.research.google.com/github/hungcs9326/airscope-webapp/blob/main/colab/rag-sensorlab/RAG02_SensorLab.ipynb)。


## 1. 我使用的資料

這份作品使用 **U.S. EPA《The Enhanced Air Sensor Guidebook》2022 版**，報告編號 EPA/600/R-22/213。
原文件為英文，內容包括空品基礎、量測規劃、感測器選擇、資料品質、共置評估與結果解釋。

- [官方資料說明](https://cfpub.epa.gov/si/si_public_record_report.cfm?Lab=CEMM&dirEntryId=356426)
- [原始 PDF](https://cfpub.epa.gov/si/si_public_file_download.cfm?Lab=CEMM&p_download_id=545843)
- 來源下載日期：2026-10-30；資料版本固定為 2022，沒有即時空品資料。
- 引用使用 **PDF 頁序**，不是書上印刷頁碼。PDF 文字直接抽取，沒有用 AI 改寫成虛構資料。
- 圖表沒有另做 OCR，部分旋轉文字、圖中數字與表格可能未完整抽取；遇到這類問題需回原 PDF 核對。
- 依 [EPA 網站著作權說明](https://www.epa.gov/web-policies-and-procedures/epa-disclaimers)，本作業限非商業科學與教育用途，保留來源資訊。


## 2. 安裝與固定匯入

沿用老師的 PDF → 分塊 → Embedding → FAISS → 打包流程。保留課堂慣用四行及 RAG 四個匯入物件，舊的 LangChain 路徑改為新版相容寫法。
首次執行需安裝套件及下載多語言模型；CPU 即可，不必選 GPU。

In [ ]:
%pip -q install "numpy==2.3.5" "pandas==2.2.3" "matplotlib==3.10.8" "faiss-cpu==1.15.1" "sentence-transformers==5.7.0" "transformers==5.22.1" "huggingface-hub==1.33.0" "langchain-core==1.7.3" "langchain-community==0.4.2" "langchain-huggingface==1.2.3" "langchain-openai==1.7.3" "langchain-classic==1.0.8" "langchain-text-splitters==1.1.3" "gradio==6.31.0" "pypdf==6.10.0" "openai==3.41.0" "groq==0.37.1" "aisuite[groq]==0.2.0" "httpx[socks]>=0.28,<1" "requests>=2.32,<3"

In [ ]:
# 保留課堂慣用四行，以及 RAG 範例四個 LangChain 物件。
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# LangChain 新版拆分套件後的相容匯入路徑。
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain

import aisuite as ai
import os
try:
    from google.colab import userdata
except ImportError:
    userdata = None
import gradio as gr

import hashlib, importlib.metadata, json, re, zipfile
from datetime import datetime
from pathlib import Path
import requests
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredWordDocumentLoader
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import display, Markdown

os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ 固定套件與 RAG 套件已匯入；不會在畫面上顯示金鑰。")


## 3. 下載原文件與建立一致的 Embedding

兩份程式固定使用相同的模型版本、`passage: `／`query: ` 前綴及單位向量正規化。
E5 支援多語言，因此可用中文問題檢索英文手冊。這不是把模型重新訓練一次。

In [ ]:
SOURCE_TITLE = "The Enhanced Air Sensor Guidebook"

SOURCE_YEAR = 2022

SOURCE_URL = "https://cfpub.epa.gov/si/si_public_file_download.cfm?Lab=CEMM&p_download_id=545843"

SOURCE_RECORD_URL = "https://cfpub.epa.gov/si/si_public_record_report.cfm?Lab=CEMM&dirEntryId=356426"

COURSE_VIDEO_URL = "https://www.youtube.com/watch?v=jSNrM7hjhek"

EMBED_MODEL = "intfloat/multilingual-e5-small"

EMBED_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"

LLM_MODEL = "groq:openai/gpt-oss-120b"

CHUNK_SIZE = 1100

CHUNK_OVERLAP = 180

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_file(url, destination, expected_sha256=None, max_mb=60):
    """Download an HTTPS source, checking status, size, and an optional hash."""
    if not url.startswith("https://"):
        raise ValueError("下載網址必須使用 HTTPS。")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=(25, 180)) as response:
        response.raise_for_status()
        size = 0
        with open(partial, "wb") as stream:
            for block in response.iter_content(1024 * 1024):
                if not block:
                    continue
                size += len(block)
                if size > max_mb * 1024 * 1024:
                    raise ValueError("下載檔案超過預期大小，請核對網址。")
                stream.write(block)
    if expected_sha256 and sha256_file(partial) != expected_sha256:
        raise ValueError("下載檔案 SHA-256 不符，停止載入；請確認資料版本。")
    partial.replace(destination)
    return destination

class E5Embeddings(Embeddings):
    """Use the same pinned model, normalization, and prefixes in both notebooks."""

    def __init__(self, model_name=EMBED_MODEL, revision=EMBED_REVISION):
        self.backend = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu", "revision": revision},
            encode_kwargs={"normalize_embeddings": True, "batch_size": 24},
            show_progress=True,
        )

    def embed_documents(self, texts):
        return self.backend.embed_documents(["passage: " + text for text in texts])

    def embed_query(self, text):
        return self.backend.embed_query("query: " + text)


In [ ]:
PDF_PATH = Path("source/Enhanced_Air_Sensor_Guidebook.pdf")
SOURCE_PDF_SHA256 = "a6b5ea68a2e4dd4b55674c2b2ff78dc209549fd23f3ae5371bc25ab0c70446e3"
if not PDF_PATH.exists() or sha256_file(PDF_PATH) != SOURCE_PDF_SHA256:
    download_file(SOURCE_URL, PDF_PATH, expected_sha256=SOURCE_PDF_SHA256)
print("✅ 官方 PDF 下載與版本核對完成：", PDF_PATH.stat().st_size, "bytes")
embedding_model = E5Embeddings()


## 4. 分塊、向量化與打包

每一塊保留文件名稱、版本、原始網址、PDF 頁序與唯一片段編號。
分塊長度 1,100 字元，重疊 180 字元，並且不跨 PDF 頁面；以減少長段落被截斷、保留可追溯頁碼。
FAISS 使用正規化向量的內積作為相似度。

In [ ]:
def read_and_split(pdf_path):
    raw_pages = PyPDFLoader(
        str(pdf_path), extraction_mode="layout",
        extraction_kwargs={"layout_mode_space_vertically": False},
    ).load()
    documents = []
    for page in raw_pages:
        text = page.page_content.replace("\x00", "")
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"(?m)^[ \t]+$", "", text)
        text = re.sub(r"\n{3,}", "\n\n", text).strip()
        if len(text) < 40:
            continue
        documents.append(Document(page_content=text, metadata={
            "source_title": SOURCE_TITLE,
            "source_url": SOURCE_URL,
            "record_url": SOURCE_RECORD_URL,
            "edition_year": SOURCE_YEAR,
            "pdf_page": int(page.metadata["page"]) + 1,
        }))
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    for number, document in enumerate(chunks, 1):
        document.metadata["chunk_id"] = f"EPA-{number:04d}"
    return raw_pages, chunks

def build_database(pdf_path, output_dir, embedding=None):
    output_dir = Path(output_dir)
    store_dir = output_dir / "faiss_db"
    store_dir.mkdir(parents=True, exist_ok=True)
    pages, chunks = read_and_split(pdf_path)
    embedding = embedding or E5Embeddings()
    vectorstore = FAISS.from_documents(
        chunks, embedding, distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
    )
    faiss.write_index(vectorstore.index, str(store_dir / "index.faiss"))
    records = []
    for row in range(vectorstore.index.ntotal):
        identifier = vectorstore.index_to_docstore_id[row]
        document = vectorstore.docstore.search(identifier)
        records.append({"id": identifier, "text": document.page_content,
                        "metadata": document.metadata})
    (store_dir / "documents.json").write_text(
        json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (output_dir / "source_text.txt").write_text(
        "\n\n".join(f"[PDF page {i+1}]\n{p.page_content}" for i, p in enumerate(pages)),
        encoding="utf-8",
    )
    manifest = {
        "schema_version": 1,
        "project": "AirScope SensorLab",
        "source_title": SOURCE_TITLE,
        "source_url": SOURCE_URL,
        "source_record_url": SOURCE_RECORD_URL,
        "source_edition": SOURCE_YEAR,
        "source_pdf_sha256": sha256_file(pdf_path),
        "created_utc": datetime.now().astimezone().isoformat(),
        "embedding_model": EMBED_MODEL,
        "embedding_revision": EMBED_REVISION,
        "document_prefix": "passage: ",
        "query_prefix": "query: ",
        "normalized": True,
        "metric": "inner_product_on_unit_vectors",
        "pdf_pages": len(pages),
        "chunks": len(records),
        "dimension": vectorstore.index.d,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "page_numbering": "1-based physical PDF page, not printed book page",
        "extraction_limit": "PDF text only; figures are not OCR-transcribed.",
        "packages": {name: importlib.metadata.version(name) for name in
                     ["faiss-cpu", "sentence-transformers", "langchain-community",
                      "langchain-huggingface", "pypdf"]},
        "files": {},
    }
    for name in ["faiss_db/index.faiss", "faiss_db/documents.json", "source_text.txt"]:
        path = output_dir / name
        manifest["files"][name] = {"sha256": sha256_file(path), "bytes": path.stat().st_size}
    (output_dir / "manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    archive = output_dir / "rag-data.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
        for name in ["manifest.json", *manifest["files"]]:
            zipped.write(output_dir / name, arcname=name)
    return vectorstore, manifest, archive


In [ ]:
vectorstore, manifest, zip_path = build_database(PDF_PATH, "rag_build", embedding_model)
display(pd.DataFrame([{
    "PDF 頁數": manifest["pdf_pages"], "文字片段數": manifest["chunks"],
    "向量維度": manifest["dimension"], "壓縮檔 bytes": zip_path.stat().st_size,
}]))
print("✅ 已建立真正的向量資料庫：", zip_path)
print("ZIP SHA-256：", sha256_file(zip_path))


## 5. 確認 ZIP 不是只有原文

`rag-data.zip` 包含 FAISS 索引、片段文字與來源 metadata、版本與雜湊紀錄、原文文字快照。
採用 FAISS 原生索引加 JSON，不依賴 pickle；RAG02 載入時會驗證內容。

In [ ]:
with zipfile.ZipFile(zip_path) as zipped:
    display(pd.DataFrame([{"檔案": x.filename, "解壓縮 bytes": x.file_size} for x in zipped.infolist()]))
print("✅ 向量筆數：", vectorstore.index.ntotal)
print("Embedding：", manifest["embedding_model"], manifest["embedding_revision"])


## 6. 下載備份與接續 RAG02

下格下載你剛建立的 ZIP。作業附的 RAG02 已填入可公開下載的預建資料庫網址，所以助教不需要手動找檔案。
若你更換資料或重新發布 ZIP，請同步修改 RAG02 的網址與 SHA-256。

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("本機產出位置：", zip_path.resolve())


## AI 協作與理解

本作業使用 ChatGPT 協助選擇環工文件、草擬人設、撰寫程式、整理測試方法與修正相容性。
主題與功能是 AI 協作提案，請本人完整讀過、確認並親自修改後再繳交。

**本次真實請求摘要（不是逐字稿）：**依老師 RAG01／RAG02 範例，選擇文件、建立可下載的向量資料庫，設計 Prompt 並以 Gradio 展示。

**AI 產出的設計與程式重點：**選用 EPA 空氣感測器手冊；以公開多語言 E5 取代需額外授權的 EmbeddingGemma；保留 FAISS；增加頁碼、證據、資料不足回覆與 Gradio 多輪訊息正規化。

**實際修正紀錄：**最初只用中文問題搜尋，濕度題的前四個片段未包含 humidity／moisture。之後加入明確列出的英文補充查詢，交錯選取中英文候選並去重。前後檢索結果都有保留；這是程式端的真實檢索測試，不代表本人已完成 Groq 對話實測。

**理解後的說明：**RAG01 把文件分塊並轉成向量。RAG02 也把問題轉成向量，找出相關片段，再把片段、問題和人設傳給 LLM。這能提供可追溯的作答依據，但檢索相似度不是正確率，答案仍需要人工核對。

📸 請附上本次 ChatGPT 的真實 Prompt 與生成結果截圖；這段摘要不能冒充逐字聊天截圖。


📸 **RAG01 重點截圖：**截取「分塊與建庫成功」的程式／輸出，以及 ZIP 檔案清單，讓向量筆數和索引檔案可見。